In [1]:
from pathlib import Path
import mlflow, mlflow.keras, tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}")

def get_exp(name):
    if mlflow.get_experiment_by_name(name) is None:
        mlflow.create_experiment(name, artifact_location=(PROJECT_ROOT / "mlruns" / name).as_uri())
    mlflow.set_experiment(name)

get_exp("brain-tumor-classification")
FIG = PROJECT_ROOT / "reports" / "figures"
run_ids = {}

# Run 1: Feature Extraction
with mlflow.start_run(run_name="inceptionv3_feature_extraction") as r:
    mlflow.log_params({"model": "InceptionV3", "strategy": "feature_extraction", "learning_rate": 1e-3,
                       "optimizer": "Adam", "loss": "categorical_crossentropy", "batch_size": 32,
                       "img_size": 299, "dropout": 0.3, "epochs_run": 12})
    mlflow.log_metrics({"best_val_loss": 0.2638, "val_accuracy": 0.8948, "train_accuracy": 0.931,
                        "test_accuracy": 0.898, "test_precision": 0.896, "test_recall": 0.896, "test_f1": 0.895})
    for f in ["fe_learning_curves.png", "class_distribution.png", "split_counts.png"]:
        if (FIG / f).exists():
            mlflow.log_artifact(str(FIG / f))
    run_ids["fe"] = r.info.run_id

# Run 2: Fine-tuning (we also log the model itself)
with mlflow.start_run(run_name="inceptionv3_fine_tuning") as r:
    mlflow.log_params({"model": "InceptionV3", "strategy": "fine_tuning", "learning_rate": 1e-5,
                       "optimizer": "Adam", "loss": "categorical_crossentropy", "batch_size": 32,
                       "img_size": 299, "dropout": 0.3, "unfrozen_layers": 50, "epochs_run": 14})
    mlflow.log_metrics({"best_val_loss": 0.1475, "val_accuracy": 0.9536, "test_accuracy": 0.960,
                        "test_precision": 0.959, "test_recall": 0.959, "test_f1": 0.959})
    model = tf.keras.models.load_model(PROJECT_ROOT / "models" / "inceptionv3_ft.keras")
    mlflow.keras.log_model(model, name="model")
    run_ids["ft"] = r.info.run_id

print(run_ids)

2026/10/28 11:39:42 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


{'fe': '822ee23df619437aab0963de67080725', 'ft': '19e2e372339f4a2d9a3ab4bfb950cd8b'}


In [2]:
import yaml
from ultralytics import YOLO

class YoloWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.m = YOLO(context.artifacts["weights"])
    def predict(self, context, model_input, params=None):
        return self.m.predict(model_input)

YOLO_DIR = PROJECT_ROOT / "reports" / "yolo"
args = yaml.safe_load(open(YOLO_DIR / "args.yaml"))
keep = ["model", "epochs", "imgsz", "batch", "patience", "seed", "hsv_h", "hsv_s", "hsv_v",
        "fliplr", "flipud", "degrees", "translate", "scale", "mosaic", "close_mosaic", "optimizer", "lr0"]

get_exp("brain-tumor-detection")
with mlflow.start_run(run_name="yolov8s_50epochs") as r:
    mlflow.log_params({k: args[k] for k in keep if k in args})
    mlflow.log_metrics({
        "valid_precision": 0.956, "valid_recall": 0.952, "valid_mAP50": 0.970, "valid_mAP50_95": 0.786,
        "test_precision": 0.956, "test_recall": 0.931, "test_mAP50": 0.959, "test_mAP50_95": 0.749,
    })
    for f in YOLO_DIR.glob("*"):
        if f.is_file():
            mlflow.log_artifact(str(f))
    mlflow.pyfunc.log_model(name="model", python_model=YoloWrapper(),
                            artifacts={"weights": str(PROJECT_ROOT / "models" / "best.pt")})
    run_ids["yolo"] = r.info.run_id

print(run_ids)

C:\Users\ycode\Desktop\brain-tumor-project\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/28 11:42:26 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


{'fe': '822ee23df619437aab0963de67080725', 'ft': '19e2e372339f4a2d9a3ab4bfb950cd8b', 'yolo': '191a961a9cad4a618052a31ec6641356'}


In [3]:
from mlflow import MlflowClient
client = MlflowClient()

# Compare runs
cls = mlflow.search_runs(experiment_names=["brain-tumor-classification"], order_by=["metrics.test_accuracy DESC"])
print(cls[["tags.mlflow.runName", "params.strategy", "params.learning_rate",
           "metrics.test_accuracy", "metrics.test_f1"]])

# Register the best classifier (fine-tuning) and the detector
def register(run_id, name, desc):
    mv = mlflow.register_model(f"runs:/{run_id}/model", name)
    client.update_model_version(name, mv.version, description=desc)
    client.set_registered_model_alias(name, "champion", mv.version)
    print(f"Registered {name} v{mv.version} (alias: champion)")

register(run_ids["ft"], "brain-tumor-classifier",
         "InceptionV3 fine-tuned (last 50 layers, lr=1e-5). Test accuracy 0.960, macro F1 0.959.")
register(run_ids["yolo"], "brain-tumor-detector",
         "YOLOv8s, 50 epochs, imgsz 640. Test mAP@0.5 0.959, mAP@0.5:0.95 0.749.")

Successfully registered model 'brain-tumor-classifier'.
2026/10/28 11:42:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 19e2e372339f4a2d9a3ab4bfb950cd8b has no artifacts at artifact path 'model', registering model based on models:/m-2687bb08ea0a4d01a00c00dddd065720 instead
Created version '1' of model 'brain-tumor-classifier'.
Successfully registered model 'brain-tumor-detector'.
2026/10/28 11:42:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 191a961a9cad4a618052a31ec6641356 has no artifacts at artifact path 'model', registering model based on models:/m-a26e3bd7b43542adab654f981174ecbb instead


              tags.mlflow.runName     params.strategy params.learning_rate  \
0         inceptionv3_fine_tuning         fine_tuning                1e-05   
1         inceptionv3_fine_tuning         fine_tuning                1e-05   
2  inceptionv3_feature_extraction  feature_extraction                0.001   
3  inceptionv3_feature_extraction  feature_extraction                0.001   

   metrics.test_accuracy  metrics.test_f1  
0                  0.960            0.959  
1                  0.960            0.959  
2                  0.898            0.895  
3                  0.898            0.895  
Registered brain-tumor-classifier v1 (alias: champion)


Created version '1' of model 'brain-tumor-detector'.


Registered brain-tumor-detector v1 (alias: champion)
